# **2026 ML FALL HW3: Unconditional Image Generation**
TODO:
- Design and configure your U-Net architecture
- Tune the training hyperparameters
- Visualize the denoising process using DDPM sampling
- Generate images using DDIM sampling with the trained diffusion model
- Compute the Fréchet Inception Distance (FID)

**Version 2: continued training.** Training continues from the 50-epoch checkpoint (`last.pt`) to 150 epochs in total, with dropout 0.1 and a fresh warm-up and cosine learning-rate phase. DDIM sampling uses `"trailing"` timestep spacing so it starts from pure noise (t = 999).

**Summary of improvements over the sample code**
- U-Net follows the original DDPM CIFAR-10 design: channels (128, 256, 256, 256), 2 ResNet layers per block, self-attention at 16×16 (~35.7M parameters).
- Random horizontal-flip augmentation (as in the DDPM paper).
- Linear warm-up + cosine learning-rate decay, AdamW without weight decay.
- EMA with warm-up (`use_ema_warmup=True`) so the EMA weights are useful after a short (50-epoch) run instead of being dominated by the random initialisation.
- Mixed-precision (fp16) training/sampling on CUDA for ~2× speed-up on a Colab T4.
- Resumable checkpoints (model, optimizer, LR scheduler, grad scaler, EMA, loss history) saved to Google Drive.
- Dropout 0.1 in the ResNet blocks (as in DDPM on CIFAR-10), to regularise longer training.
- DDIM with `timestep_spacing="trailing"` (first step at t = 999 instead of 990).
- FID computed with `pytorch-fid` on 2048-d Inception pool features.

# **References**

[Denoising Diffusion Probabilistic Models (DDPM)](https://arxiv.org/abs/2006.11239)

[Denoising Diffusion Implicit Models (DDIM)](https://arxiv.org/abs/2010.02502)

# **Download Packages and Dataset**

In [ ]:
%pip uninstall -y torchao
%pip install -q diffusers accelerate pytorch-fid gdown

In [ ]:
!gdown 1S5gv9hOBDAoENUvwTYCXfCvTfXeg6JaW
!unzip -q -o HW3.zip

# **Import Packages**

In [ ]:
import math
import random
import torch
import shutil
import time
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F

from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from google.colab import drive

from diffusers import UNet2DModel, DDPMScheduler, DDIMScheduler
from diffusers.training_utils import EMAModel

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import make_grid
from torchvision.datasets import CIFAR10

from pytorch_fid.fid_score import calculate_fid_given_paths

# **Metadata**

In [ ]:
drive.mount("/content/drive")
OUTPUT_DIR = Path("/content/drive/MyDrive/HW3/ddpm")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# TODO: put your student ID here (first letter lowercase), used for the submission file name.
STUDENT_ID = "d15949002"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", DEVICE)

# Mixed precision (fp16) is only used on CUDA GPUs.
USE_AMP = DEVICE.type == "cuda"
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True
    print("GPU:", torch.cuda.get_device_name(0))

# **Dataset**

In [ ]:
# Adjust these settings according to your available computational resources.
BATCH_SIZE = 128
NUM_WORKERS = 2

In [ ]:
DATA_DIR = Path("data/train")

class PNGDataset(Dataset):
    def __init__(self, directory, augment=True):
        self.files = sorted(Path(directory).glob("*.png"))
        aug = [transforms.RandomHorizontalFlip()] if augment else []
        self.transform = transforms.Compose(aug + [
            transforms.ToTensor(),
            transforms.Normalize((0.5,)*3, (0.5,)*3),
        ])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, index):
        with Image.open(self.files[index]) as img:
            return self.transform(img.convert("RGB"))

dataset = PNGDataset(DATA_DIR, augment=True)
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    drop_last=True,
    persistent_workers=(NUM_WORKERS > 0),
)

print(f"Training images: {len(dataset)}; Shape: {tuple(dataset[0].shape)}")

# **Model**

In [ ]:
# U-Net for noise prediction, following the DDPM CIFAR-10 configuration (Ho et al., 2020).
# Resolutions of the 4 levels: 32x32 -> 16x16 -> 8x8 -> 4x4
BLOCK_OUT_CHANNELS = (128, 256, 256, 256)
LAYERS_PER_BLOCK = 2

DOWN_BLOCK_TYPES = (
    "DownBlock2D",       # 32x32, 128 ch
    "AttnDownBlock2D",   # 16x16, 256 ch + self-attention
    "DownBlock2D",       # 8x8,   256 ch
    "DownBlock2D",       # 4x4,   256 ch (no further downsampling)
)
UP_BLOCK_TYPES = (
    "UpBlock2D",         # 4x4
    "UpBlock2D",         # 8x8
    "AttnUpBlock2D",     # 16x16 + self-attention
    "UpBlock2D",         # 32x32
)

DROPOUT = 0.1

# ---- Training plan ----
# "continue": load the 50-epoch run (OUTPUT_DIR/last.pt) and keep training up to EPOCHS in total.
# "scratch" : train a new model from random initialisation for EPOCHS epochs.
TRAIN_MODE = "continue"
INIT_CHECKPOINT = "last.pt"       # checkpoint of the first (50-epoch) run
RUN_CHECKPOINT = "last_v2.pt"     # checkpoint of this run (so the first run is never overwritten)

EPOCHS = 150             # total epochs (including the 50 already trained when TRAIN_MODE="continue")
LEARNING_RATE = 2e-4     # peak LR of this run's schedule
WEIGHT_DECAY = 0.0
WARMUP_STEPS = 1000
MIN_LR_RATIO = 0.05      # cosine decays the LR to 5% of its peak
GRAD_CLIP = 1.0

In [ ]:
DIFFUSION_STEPS = 1000

model = UNet2DModel(
    sample_size=32,
    in_channels=3,
    out_channels=3,
    layers_per_block=LAYERS_PER_BLOCK,
    block_out_channels=BLOCK_OUT_CHANNELS,
    down_block_types=DOWN_BLOCK_TYPES,
    up_block_types=UP_BLOCK_TYPES,
    norm_num_groups=32,
    dropout=DROPOUT,
).to(DEVICE)

# EMA with warm-up: decay = 1 - (1 + step)^(-2/3), capped at 0.9999.
# With ~20k training steps a fixed 0.9999 decay would still carry a lot of the early,
# poorly trained weights; the warm-up makes the EMA track the model properly.
ema_model = EMAModel(
    model.parameters(),
    decay=0.9999,
    use_ema_warmup=True,
    inv_gamma=1.0,
    power=2 / 3,
)

noise_scheduler = DDPMScheduler(
    num_train_timesteps=DIFFUSION_STEPS,
    beta_schedule="linear",
    prediction_type="epsilon",
)

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.999)
)

scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Steps per epoch: {len(loader)}")

# **Train**

In [ ]:
# If this run's own checkpoint (last_v2.pt) exists, training automatically resumes from it,
# e.g. after a Colab disconnect: just re-run all cells.
checkpoint_path = OUTPUT_DIR / RUN_CHECKPOINT
init_path = OUTPUT_DIR / INIT_CHECKPOINT

start_epoch = 0          # epoch to continue from
phase_start_epoch = 0    # epoch at which this run's LR schedule starts
loss_history = []
resume_state = None

def load_weights(state):
    model.load_state_dict(state["model"])
    optimizer.load_state_dict(state["optimizer"])
    ema_model.load_state_dict(state["ema"])
    if "scaler" in state:
        scaler.load_state_dict(state["scaler"])

if checkpoint_path.exists():
    resume_state = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    load_weights(resume_state)
    start_epoch = resume_state["epoch"]
    phase_start_epoch = resume_state["phase_start_epoch"]
    loss_history = resume_state.get("loss_history", [])
    print(f"Resuming this run from epoch {start_epoch}")
elif TRAIN_MODE == "continue":
    assert init_path.exists(), f"{init_path} not found - set TRAIN_MODE = 'scratch'"
    state = torch.load(init_path, map_location=DEVICE, weights_only=False)
    load_weights(state)
    start_epoch = phase_start_epoch = state["epoch"]
    loss_history = state.get("loss_history", [])
    print(f"Continuing from the first run's checkpoint at epoch {start_epoch}")
    del state
else:
    print("Training from scratch")

# A new LR schedule for this run: linear warm-up, then cosine decay until EPOCHS.
for group in optimizer.param_groups:
    group["lr"] = LEARNING_RATE
    group["initial_lr"] = LEARNING_RATE

PHASE_STEPS = max(1, (EPOCHS - phase_start_epoch) * len(loader))

def lr_lambda(step):
    if step < WARMUP_STEPS:
        return (step + 1) / WARMUP_STEPS
    progress = min(1.0, (step - WARMUP_STEPS) / max(1, PHASE_STEPS - WARMUP_STEPS))
    return MIN_LR_RATIO + (1 - MIN_LR_RATIO) * 0.5 * (1 + math.cos(math.pi * progress))

lr_scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
if resume_state is not None:
    lr_scheduler.load_state_dict(resume_state["lr_scheduler"])
    del resume_state

print(f"Training epochs {start_epoch+1}-{EPOCHS} ({PHASE_STEPS} scheduled steps in this run)")

for epoch in range(start_epoch, EPOCHS):
    model.train()
    total_loss = 0.0
    seen = 0
    bar = tqdm(loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for clean_images in bar:
        clean_images = clean_images.to(DEVICE, non_blocking=True)
        noise = torch.randn_like(clean_images)

        timesteps = torch.randint(
            0,
            DIFFUSION_STEPS,
            (clean_images.size(0),),
            device=DEVICE,
        ).long()

        noisy_images = noise_scheduler.add_noise(
            clean_images,
            noise,
            timesteps,
        )

        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            predicted_noise = model(noisy_images, timesteps).sample
        loss = F.mse_loss(predicted_noise.float(), noise)

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        lr_scheduler.step()
        ema_model.step(model.parameters())

        total_loss += loss.item() * clean_images.size(0)
        seen += clean_images.size(0)
        bar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{lr_scheduler.get_last_lr()[0]:.2e}")

    avg_loss = total_loss / seen
    loss_history.append(avg_loss)
    print(f"Epoch {epoch+1}: mean MSE = {avg_loss:.6f}")

    torch.save(
        {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "ema": ema_model.state_dict(),
            "lr_scheduler": lr_scheduler.state_dict(),
            "scaler": scaler.state_dict(),
            "epoch": epoch+1,
            "phase_start_epoch": phase_start_epoch,
            "loss_history": loss_history,
            "model_config": dict(model.config),
            "scheduler_config": dict(noise_scheduler.config),
        },
        checkpoint_path,
    )

print("Checkpoint:", checkpoint_path)

In [ ]:
# Training loss curve (useful for the report)
if loss_history:
    plt.figure(figsize=(6, 4))
    first = EPOCHS - len(loss_history) + 1
    plt.plot(range(first, EPOCHS + 1), loss_history, marker="o", ms=3)
    if TRAIN_MODE == "continue" and phase_start_epoch > 0:
        plt.axvline(phase_start_epoch + 0.5, color="gray", ls="--", lw=1, label="continued training")
        plt.legend()
    plt.xlabel("Epoch"); plt.ylabel("Mean MSE"); plt.title("Training loss")
    plt.grid(alpha=0.3)
    plt.savefig("training_loss.png", dpi=200, bbox_inches="tight")
    plt.show()

In [ ]:
# Save the EMA weights (used for all sampling) as the submission model: <student_id>_hw3.pt
ema_model.store(model.parameters())
ema_model.copy_to(model.parameters())
submission_path = OUTPUT_DIR / f"{STUDENT_ID}_hw3.pt"
torch.save(
    {
        "model": model.state_dict(),
        "model_config": dict(model.config),
        "scheduler_config": dict(noise_scheduler.config),
    },
    submission_path,
)
ema_model.restore(model.parameters())
print("Saved submission model to", submission_path)

# **Denoising Process**

In [ ]:
# Run with two different sampling step counts and compare the results in the report.
SAMPLE_STEPS_LIST = [100, 1000]

In [ ]:
SAMPLE_SEED = 2026
SAMPLE_BATCH_SIZE = 64
VIS_IMAGES = 4

def visualize_denoising(SAMPLE_STEPS):
    VIS_STEPS = sorted(set(
        round(step * SAMPLE_STEPS / 1000)
        for step in [
            0, 100, 200, 300, 400, 500,
            600, 700, 750, 800, 850,
            900, 925, 950, 975, 1000
        ]
    ))

    model.eval()
    ema_model.store(model.parameters())
    ema_model.copy_to(model.parameters())

    noise_scheduler.set_timesteps(SAMPLE_STEPS, device=DEVICE)
    generator = torch.Generator(device=DEVICE).manual_seed(SAMPLE_SEED)

    x = torch.randn((VIS_IMAGES, 3, 32, 32), generator=generator, device=DEVICE)
    snapshots = {0: x.detach().cpu().clone()}

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    elif DEVICE.type == "mps":
        torch.mps.synchronize()
    start_time = time.perf_counter()

    with torch.inference_mode():
        for step, t in enumerate(
            tqdm(noise_scheduler.timesteps, desc=f"Denoising ({SAMPLE_STEPS} steps)"),
            start=1
        ):
            eps = model(x, t).sample
            x = noise_scheduler.step(
                eps, t, x, generator=generator
            ).prev_sample

            if step in VIS_STEPS:
                snapshots[step] = x.detach().cpu().clone()

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    elif DEVICE.type == "mps":
        torch.mps.synchronize()
    elapsed_time = time.perf_counter() - start_time

    print(f"Sampling steps: {SAMPLE_STEPS}")
    print(f"Sampling time: {elapsed_time:.2f} seconds")
    print(f"Time per step: {elapsed_time / SAMPLE_STEPS:.4f} seconds")

    ema_model.restore(model.parameters())

    fig, axes = plt.subplots(VIS_IMAGES, len(VIS_STEPS), figsize=(22, 8))
    fig.suptitle(
        f"DDPM Denoising Process ({SAMPLE_STEPS} Sampling Steps)",
        fontsize=16, fontweight="bold", y=1.02,
    )

    for col, step in enumerate(VIS_STEPS):
        images = (snapshots[step].clamp(-1, 1) + 1) / 2

        for row in range(VIS_IMAGES):
            axes[row, col].imshow(
                images[row].permute(1, 2, 0).numpy()
            )
            axes[row, col].axis("off")

            if row == 0:
                axes[row, col].set_title(f"Step {step}")

    plt.tight_layout()
    plt.savefig(f"denoising_process_{SAMPLE_STEPS}.png", dpi=200, bbox_inches="tight")
    plt.show()
    return elapsed_time

sampling_times = {}
for steps in SAMPLE_STEPS_LIST:
    sampling_times[steps] = visualize_denoising(steps)

print("\nSummary:")
for steps, sec in sampling_times.items():
    print(f"  {steps:>4} steps: {sec:.2f} s ({sec/steps:.4f} s/step)")

# **Image Generation**

In [ ]:
SAMPLE_STEPS = 100
GEN_BATCH_SIZE = 500   # larger batches = faster sampling (32x32 images fit easily in memory)

PREVIEW_IMAGES = 16
FID_IMAGES = 10000

NUM_GENERATE = FID_IMAGES
GENERATED_DIR = Path("/content/generated")

if GENERATED_DIR.exists():
    shutil.rmtree(GENERATED_DIR)
GENERATED_DIR.mkdir(parents=True)

model.eval()

ema_model.store(model.parameters())
ema_model.copy_to(model.parameters())

# "trailing" spacing: timesteps 999, 989, ..., 9 (the default "leading" spacing starts at 990).
sampling_scheduler = DDIMScheduler.from_config(noise_scheduler.config, timestep_spacing="trailing")
sampling_scheduler.set_timesteps(SAMPLE_STEPS, device=DEVICE)
print("DDIM timesteps:", sampling_scheduler.timesteps[:3].tolist(), "...", sampling_scheduler.timesteps[-3:].tolist())
generator = torch.Generator(device=DEVICE).manual_seed(SAMPLE_SEED)

preview = []
gen_start = time.perf_counter()
with torch.inference_mode():
    for start in tqdm(range(0, NUM_GENERATE, GEN_BATCH_SIZE), desc="Sampling"):
        n = min(GEN_BATCH_SIZE, NUM_GENERATE-start)
        x = torch.randn((n, 3, 32, 32), generator=generator, device=DEVICE)

        for t in sampling_scheduler.timesteps:
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                eps = model(x, t).sample
            x = sampling_scheduler.step(
                eps.float(),
                t,
                x,
                eta=0.0,
            ).prev_sample

        x = ((x.clamp(-1, 1) + 1) / 2).cpu()
        for j, img in enumerate(x):
            transforms.ToPILImage()(img).save(GENERATED_DIR / f"{start+j:05d}.png")

        if len(preview) < PREVIEW_IMAGES:
            preview.extend(list(x[:max(0, PREVIEW_IMAGES-len(preview))]))

ema_model.restore(model.parameters())

print("Saved", NUM_GENERATE, "images to", GENERATED_DIR, f"in {time.perf_counter()-gen_start:.1f} s")
if preview:
    grid = make_grid(torch.stack(preview), nrow=4)
    plt.figure(figsize=(7, 7))
    plt.imshow(grid.permute(1, 2, 0))
    plt.title(
        f"DDIM Generated Images ({SAMPLE_STEPS} Steps)",
        fontsize=16,
        fontweight="bold",
        pad=15
    )
    plt.axis("off")
    plt.savefig("generated_images.png", dpi=200, bbox_inches="tight")
    plt.show()

# **FID Evaluation**

In [ ]:
REAL_DIR = Path("data/test")
print("Reference images:", len(list(REAL_DIR.glob("*.png"))))

In [ ]:
def compute_fid(image_dir):
    # FID between `image_dir` and the CIFAR-10 test set, using 2048-d Inception-v3 pool3 features.
    return calculate_fid_given_paths(
        [str(image_dir), str(REAL_DIR)],
        batch_size=200,
        device=DEVICE,
        dims=2048,
        num_workers=NUM_WORKERS,
    )

fid_generated = compute_fid(GENERATED_DIR)
fid_train = compute_fid(DATA_DIR)

print(f"FID ({FID_IMAGES} generated vs 10000 test): {fid_generated:.4f}")
print(f"FID (50000 train vs 10000 test): {fid_train:.4f}")